# Talk-openpyxl
## Reading and writing Microsoft Excel Files with openpyxl

![](images/AutomateTheBoringStuff.png)

https://automatetheboringstuff.com/

https://nostarch.com/automate-boring-stuff-python-3rd-edition

* Introducing one of the handiest Python books
* 2nd edition free to read online
* 1st part is learn to program, 2nd part shows how to use Python to automate common tasks
* Most of what I've learned about openpyxl is from the "Working with Excel Spreadsheets"
* 3rd edition adds chapters on: Google Sheets, SQLite Databases, Making graphs, Recognizing Text in images, Text to speech and Speech Recognition Engines

## 1 - Open an Excel Workbook

In [1]:
from openpyxl import load_workbook

wb_in = load_workbook('RuralAtlasData24.xlsx')
wb_in

In [2]:
# Attempt to load something that isn't an Excel workbook
# catch the exception

from sys import stderr
from openpyxl.utils.exceptions import InvalidFileException

try:
    load_workbook('FoodInspectionsFirst100.csv')
except InvalidFileException as e:
    print(e, file=stderr)

openpyxl does not support .csv file format, please check you can open it with Excel first. Supported formats are: .xlsx,.xlsm,.xltx,.xltm


In [3]:
# This can also raise FileNotFound and other standard I/O errors
try:
    load_workbook('notthere.xlsx')
except FileNotFoundError as e:
    print(e, file=stderr)

[Errno 2] No such file or directory: 'notthere.xlsx'


## 2 - Read data from a Workbook

### List Worksheets

In [4]:
wb_in.sheetnames

['Read Me',
 'VariableNameLookup',
 'Documentation',
 'People',
 'Jobs',
 'County Classifications',
 'Income',
 'Veterans']

### Get active sheet

In [5]:
# note this also a writeable property
wb_in.active

<Worksheet "Read Me">

### Access Worksheet, by title

In [6]:
# You can think of a Workbook as dict of Worksheets
sheet = wb_in['VariableNameLookup']
sheet

<Worksheet "VariableNameLookup">

### Worksheet title

In [7]:
sheet.title  

'VariableNameLookup'

### Worksheet boundaries max row and column

In [8]:
# boundaries of the sheet max_row and max_column
print(f"{sheet.max_row=}")
print(f"{sheet.max_column=}")

sheet.max_row=250
sheet.max_column=5


In [9]:
# sheet.columns and sheet.rows are generators
print(f"{sheet.columns=}")
print(f"{sheet.rows=}")

sheet.columns=<generator object Worksheet._cells_by_col at 0x710a600a9900>
sheet.rows=<generator object Worksheet._cells_by_row at 0x710a600a8ee0>


In [10]:
# sheet.columns (2D array - cells grouped by columns)
[c for c in sheet.columns][0][:4]

(<Cell 'VariableNameLookup'.A1>,
 <Cell 'VariableNameLookup'.A2>,
 <Cell 'VariableNameLookup'.A3>,
 <Cell 'VariableNameLookup'.A4>)

In [11]:
# sheet.rows (2d array - cells grouped by rows)
[c for c in sheet.rows][1][3:]

(<Cell 'VariableNameLookup'.D2>, <Cell 'VariableNameLookup'.E2>)

In [12]:
# simple iteration by rows
for i, row in enumerate(sheet.rows, 1):
    for j, cell in enumerate(row, 1):
        print(cell.value)
        if j >= 2:
            break
    if i >= 3:
        break

Cat_Sort
Category
1
People
1
People


### Get a single cell `Worksheet.cell(row, col)`

In [13]:
# Note row and column start at 1 not 0!!!
sheet.cell(1, 2)

<Cell 'VariableNameLookup'.B1>

### Get a single cell via `Worksheet['B1']`

In [14]:
cell = sheet['B1']

### Cell Properties

In [15]:
print(f"{cell.col_idx=}")
print(f"{cell.column=}")
print(f"{cell.column_letter=}")
print(f"{cell.row=}")
print(f"{cell.value=}")
print(f"{cell.hyperlink=}")

cell.col_idx=2
cell.column=2
cell.column_letter='B'
cell.row=1
cell.value='Category'
cell.hyperlink=None


In [16]:
# 2D array - cells grouped by row
selection = sheet['A1':'E2']
selection

((<Cell 'VariableNameLookup'.A1>,
  <Cell 'VariableNameLookup'.B1>,
  <Cell 'VariableNameLookup'.C1>,
  <Cell 'VariableNameLookup'.D1>,
  <Cell 'VariableNameLookup'.E1>),
 (<Cell 'VariableNameLookup'.A2>,
  <Cell 'VariableNameLookup'.B2>,
  <Cell 'VariableNameLookup'.C2>,
  <Cell 'VariableNameLookup'.D2>,
  <Cell 'VariableNameLookup'.E2>))

In [17]:
# part of 1 column selected still cells grouped by rows
sheet['A1':'A3']

((<Cell 'VariableNameLookup'.A1>,),
 (<Cell 'VariableNameLookup'.A2>,),
 (<Cell 'VariableNameLookup'.A3>,))

In [18]:
# part of 1 row selected, still get tuple of tuples
sheet['A1':'C1']

((<Cell 'VariableNameLookup'.A1>,
  <Cell 'VariableNameLookup'.B1>,
  <Cell 'VariableNameLookup'.C1>),)

### Converting between numeric and string column indexes

In [35]:
from openpyxl.utils import (
    column_index_from_string,
    get_column_letter,
)
int_col = 27
print(f"{column_index_from_string('AA')=}")
print(f"{get_column_letter(27)=}")

column_index_from_string('AA')=27
get_column_letter(27)='AA'


## 3 - Create a Workbook

In [22]:
from openpyxl import Workbook

wb_out = Workbook()

In [25]:
# By default workbook has 1 worksheet
wb_out.sheetnames

['Sheet']

In [29]:
# Get the active sheet (note this also a writeable property)
sheet = wb_out.active
sheet

<Worksheet "Sheet">

In [30]:
# Change the title of the sheet and check sheetnames
sheet.title = "README"
wb_out.sheetnames

['README']

In [31]:
# Add a sheet to the workbook
failed_sheet = wb_out.create_sheet('Failed')
wb_out.sheetnames

['README', 'Failed']

In [33]:
# insert a sheet at specified index
passed_sheet = wb_out.create_sheet('Passed', index=1)
wb_out.sheetnames

['README', 'Passed', 'Failed']

In [34]:
# Make Passed the active worksheet
wb_out.active = passed_sheet
wb_out.active

<Worksheet "Passed">

In [11]:
# Make Passed the active worksheet
wb.active = passed_sheet
wb.active

<Worksheet "Passed">

In [12]:
# add another Worksheet so we can move it then delete it
delete_me = wb.create_sheet('DELETEME')
wb.sheetnamesx

['README', 'Passed', 'Failed', 'DELETEME']

In [13]:
wb.move_sheet(delete_me, offset=-1)
wb.sheetnames

['README', 'Passed', 'DELETEME', 'Failed']

In [14]:
# Now remove the Worksheet
wb.remove(delete_me)
wb.sheetnames

['README', 'Passed', 'Failed']

In [15]:
# You'll get a ValueError if the sheet isn't present
try:
    wb.remove(delete_me)
except ValueError:
    print(f"{delete_me.title} not present might as well delete the variable", file=stderr)
    del delete_me

DELETEME not present might as well delete the variable
